In [ ]:
# ══════════════════════════════════════════════════════════════
#  RUN MODE — change this before running
#  'attack_only' : runs setup + attack generation only (~6 hrs)
#                  saves adversarial_examples.csv + augmented_training_set.csv
#  'full'        : runs everything (use after attack CSVs are saved)
# ══════════════════════════════════════════════════════════════
RUN_MODE = 'full'   # ← change to 'full' for complete run

import os, subprocess

REPO_DIR = 'SRTY3009-IODetection'
if not os.path.isfile('test_set.csv') and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Banjomenny/SRTY3009-IODetection.git'], check=True)

DATA_ROOT = REPO_DIR if os.path.isdir(REPO_DIR) else '.'

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:    {device}')
print(f'Run mode:  {RUN_MODE}')

MODEL_ID = 'Banjomenny/DisInfoBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

print(f'Model loaded from {MODEL_ID}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Data root:  {DATA_ROOT}')


In [ ]:
%%time
if RUN_MODE != 'full':
    print('Skipping: Section 1 baseline eval')
else:
    import pandas as pd
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    from sklearn.metrics import f1_score, accuracy_score
    import matplotlib.pyplot as plt
    
    test_df = pd.read_csv(os.path.join(DATA_ROOT, 'test_set.csv'))
    X_test_text = test_df['text']
    try:
        y_test = test_df['Label']
    except KeyError:
        y_test = test_df['label']
    
    print(f'Test set loaded: {len(X_test_text):,} rows')
    print(f'  Organic: {(y_test == 0).sum():,}  |  IO: {(y_test == 1).sum():,}')
    
    print('\nRunning inference on test set...')
    BATCH_SIZE = 256
    all_preds = []
    total_batches = (len(X_test_text) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(0, len(X_test_text), BATCH_SIZE):
        batch_texts = X_test_text[i:i+BATCH_SIZE].tolist()
        inputs = tokenizer(
            batch_texts, truncation=True, padding=True,
            max_length=128, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        batch_num = i // BATCH_SIZE + 1
        if batch_num % 25 == 0 or batch_num == total_batches:
            print(f'  Batch {batch_num}/{total_batches}')
    
    y_pred = np.array(all_preds)
    
    print()
    print('=' * 55)
    print('  SECTION 1 — CLASSIFICATION REPORT')
    print('  DisInfoBERT (Fine-tuned Twitter-RoBERTa)')
    print('=' * 55)
    print()
    print(classification_report(y_test, y_pred, target_names=['Organic', 'IO'], digits=4))
    
    roberta_f1  = f1_score(y_test, y_pred, average='weighted')
    roberta_acc = accuracy_score(y_test, y_pred)
    print(f'  Weighted F1:  {roberta_f1:.4f}')
    print(f'  Accuracy:     {roberta_acc:.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('DisInfoBERT — Confusion Matrix', fontsize=14)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Organic', 'IO']).plot(ax=axes[0], cmap='Blues', values_format=',')
    axes[0].set_title('Raw Counts')
    cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
    ConfusionMatrixDisplay(cm_norm, display_labels=['Organic', 'IO']).plot(ax=axes[1], cmap='Blues', values_format='.3f')
    axes[1].set_title('Normalized')
    plt.tight_layout()
    plt.savefig('section1_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: section1_confusion_matrix.png')


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 2 hyperparams')
else:
    # Section 2 — Fine-Tuning Results
    # Hyperparameters used, text preprocessing kept/dropped, before vs after comparison
    
    # ── Training hyperparameters ─────────────────────────────
    print('=' * 60)
    print('  SECTION 2 — FINE-TUNING & IMPROVEMENTS')
    print('=' * 60)
    
    print('\n  Base model:  cardiffnlp/twitter-roberta-base (125M params)')
    print('  Task:        Binary classification (Organic=0, IO=1)')
    print('  Tokenizer:   RoBERTa BPE, max_length=128')
    
    print('\n  Training hyperparameters:')
    print('  ┌─────────────────────────┬────────────┐')
    print('  │ Hyperparameter          │ Value      │')
    print('  ├─────────────────────────┼────────────┤')
    print('  │ Learning rate           │ 2e-5       │')
    print('  │ Batch size (train)      │ 16         │')
    print('  │ Batch size (eval)       │ 32         │')
    print('  │ Epochs                  │ 3          │')
    print('  │ Optimizer               │ AdamW      │')
    print('  │ FP16 mixed precision    │ Yes        │')
    print('  │ Metric for best model   │ F1 (wt.)   │')
    print('  │ Evaluation strategy     │ per epoch  │')
    print('  │ Seed                    │ 42         │')
    print('  └─────────────────────────┴────────────┘')
    
    # ── Text preprocessing kept vs dropped ───────────────────
    print('\n  Text preprocessing (NLP equivalent of feature engineering):')
    print('  ┌──────────────────────────────┬──────────┬─────────────────────────────┐')
    print('  │ Preprocessing Step           │ Kept?    │ Reason                      │')
    print('  ├──────────────────────────────┼──────────┼─────────────────────────────┤')
    print('  │ URL removal                  │ Removed  │ URLs are noise, not content │')
    print('  │ @mention removal             │ Removed  │ User handles leak identity  │')
    print('  │ Hashtag removal              │ Removed  │ Hashtags vary by campaign   │')
    print('  │ Retweet prefix strip         │ Removed  │ RT markers not predictive   │')
    print('  │ Non-ASCII removal            │ Removed  │ Emoji/special chars = noise │')
    print('  │ Short text filter (>20 char) │ Applied  │ Too-short posts lack signal │')
    print('  │ Lowercasing                  │ Kept     │ RoBERTa tokenizer handles   │')
    print('  │ Punctuation                  │ Kept     │ RoBERTa uses subword tokens │')
    print('  │ Stop words                   │ Kept     │ Contextual model uses them  │')
    print('  └──────────────────────────────┴──────────┴─────────────────────────────┘')
    
    # ── Before vs After comparison ───────────────────────────
    # A1 baseline: TF-IDF + Logistic Regression (from context doc)
    baseline_f1  = 0.83
    baseline_acc = 0.83
    
    f1_change  = roberta_f1 - baseline_f1
    acc_change = roberta_acc - baseline_acc
    
    print('\n  Before vs After comparison:')
    print('  ┌──────────────────┬─────────────────────┬─────────────────────┬──────────┐')
    print('  │ Metric           │ Baseline (TF-IDF+LR)│ Fine-tuned RoBERTa  │ Change   │')
    print('  ├──────────────────┼─────────────────────┼─────────────────────┼──────────┤')
    print(f'  │ F1 (weighted)    │ {baseline_f1:.4f}              │ {roberta_f1:.4f}              │ {f1_change:+.4f}  │')
    print(f'  │ Accuracy         │ {baseline_acc:.4f}              │ {roberta_acc:.4f}              │ {acc_change:+.4f}  │')
    print(f'  │ Model type       │ Bag-of-words         │ Contextual (attention)│ upgrade  │')
    print(f'  │ Parameters       │ TF-IDF vocab (~50K)  │ 125M                 │ larger   │')
    print('  └──────────────────┴─────────────────────┴─────────────────────┴──────────┘')
    
    print(f'\n  Summary: RoBERTa {"improves" if f1_change > 0 else "underperforms"} over TF-IDF+LR baseline')
    print(f'  by {abs(f1_change):.4f} F1 points. RoBERTa captures word order and context')
    print(f'  that bag-of-words TF-IDF misses, which matters for detecting')
    print(f'  coordinated IO content that relies on framing rather than keywords.')


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 2b feature importance')
else:
    # Section 2b — Feature Importance via RoBERTa Attention Weights
    # For transformer models, attention weights show which tokens the model
    # focuses on when making predictions. We aggregate attention from the
    # last layer across 200 test samples, split by class, to find the tokens
    # most associated with IO vs Organic classification.
    
    import torch
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from collections import defaultdict
    
    # Load test set
    test_df = pd.read_csv('/kaggle/working/SRTY3009-IODetection/test_set.csv')
    test_sample = test_df.sample(n=200, random_state=42).reset_index(drop=True)
    
    # Reload model with output_attentions enabled
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    MODEL_ID = 'Banjomenny/DisInfoBERT'
    tokenizer_att = AutoTokenizer.from_pretrained(MODEL_ID)
    model_att = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, output_attentions=True
    )
    model_att.to(device)
    model_att.eval()
    
    # Aggregate token attention by class
    token_attention_io  = defaultdict(list)
    token_attention_org = defaultdict(list)
    
    print("Extracting attention weights from 200 test samples...")
    
    for _, row in test_sample.iterrows():
        inputs = tokenizer_att(
            row['text'],
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        ).to(device)
    
        with torch.no_grad():
            outputs = model_att(**inputs)
    
        # outputs.attentions: tuple of (num_layers,) each (1, num_heads, seq_len, seq_len)
        # Use last layer, average across heads, take CLS token attention (row 0)
        last_layer_attn = outputs.attentions[-1]          # (1, heads, seq, seq)
        cls_attn = last_layer_attn[0].mean(dim=0)[0]     # (seq_len,) — CLS attending to all tokens
        cls_attn = cls_attn.cpu().numpy()
    
        # Get actual tokens (skip [CLS] and [SEP] and padding)
        input_ids = inputs['input_ids'][0].cpu().numpy()
        tokens = tokenizer_att.convert_ids_to_tokens(input_ids)
    
        for tok, attn_val in zip(tokens, cls_attn):
            if tok in ['<s>', '</s>', '<pad>', 'Ġ']:
                continue
            # Strip RoBERTa's Ġ (space) prefix for cleaner labels
            clean_tok = tok.lstrip('Ġ').lower()
            if len(clean_tok) < 2:
                continue
            if row['Label'] == 1:
                token_attention_io[clean_tok].append(attn_val)
            else:
                token_attention_org[clean_tok].append(attn_val)
    
    # Mean attention per token per class
    io_means  = {t: np.mean(v) for t, v in token_attention_io.items()  if len(v) >= 5}
    org_means = {t: np.mean(v) for t, v in token_attention_org.items() if len(v) >= 5}
    
    # Discriminative score: tokens with highest IO attention relative to Organic
    all_tokens = set(io_means.keys()) & set(org_means.keys())
    diff_scores = {t: io_means[t] - org_means[t] for t in all_tokens}
    
    top_io  = sorted(diff_scores.items(), key=lambda x: x[1],  reverse=True)[:20]
    top_org = sorted(diff_scores.items(), key=lambda x: x[1])[:20]
    
    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(
        'Section 2 — Feature Importance: RoBERTa Attention Weights\n'
        'Tokens the model focuses on most (CLS attention, last layer, 200 test samples)',
        fontsize=12
    )
    
    io_labels  = [t[0] for t in top_io]
    io_scores  = [t[1] for t in top_io]
    axes[0].barh(io_labels[::-1], io_scores[::-1], color='#e05c5c')
    axes[0].set_title('Top 20 Tokens → IO Classification')
    axes[0].set_xlabel('Mean Attention Differential (IO − Organic)')
    axes[0].tick_params(axis='y', labelsize=9)
    
    org_labels = [t[0] for t in top_org]
    org_scores = [abs(t[1]) for t in top_org]
    axes[1].barh(org_labels[::-1], org_scores[::-1], color='#5c9ee0')
    axes[1].set_title('Top 20 Tokens → Organic Classification')
    axes[1].set_xlabel('Mean Attention Differential (Organic − IO)')
    axes[1].tick_params(axis='y', labelsize=9)
    
    plt.tight_layout()
    plt.savefig('section2_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: section2_feature_importance.png')
    
    # ── Summary ───────────────────────────────────────────────────────────────
    print(f'\n  Total unique tokens seen     : {len(diff_scores):,}')
    print(f'  Tokens shown (top 20 each)   : 40')
    print(f'  Tokens below frequency threshold (< 5 samples): dropped')
    print(f'  Method: CLS token attention, last transformer layer, averaged across heads')
    print(f'  These are the tokens RoBERTa attends to most when classifying each category.')


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 3 prediction demo')
else:
    # Section 3 — Prediction Pipeline Demo
    # 3 test-set examples: 1 Organic + 2 IO, with real label, predicted label, confidence
    
    def predict_post(text, real_label):
        """Run a single tweet through the fine-tuned RoBERTa model."""
        label_map = {0: 'Organic', 1: 'IO'}
        inputs = tokenizer(
            text, truncation=True, padding='max_length',
            max_length=128, return_tensors='pt'
        ).to(device)
    
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]
    
        pred_idx   = torch.argmax(probs).item()
        pred_name  = label_map[pred_idx]
        real_name  = label_map[real_label]
        confidence = probs[pred_idx].item()
        correct    = pred_idx == real_label
    
        print('=' * 60)
        print(f'  Text:       {text[:80]}{"..." if len(text) > 80 else ""}')
        print(f'  Real label: {real_name}')
        print(f'  Predicted:  {pred_name}  {"CORRECT" if correct else "INCORRECT"}')
        print(f'  Confidence: {confidence:.4f}')
        print(f'  P(Organic): {probs[0]:.4f}   P(IO): {probs[1]:.4f}')
        print('=' * 60)
        return pred_name, real_name, confidence, correct
    
    # ── Pick 1 Organic + 2 IO from the test set ──────────────
    organic_idx = y_test[y_test == 0].sample(1, random_state=7).index
    io_idx      = y_test[y_test == 1].sample(2, random_state=7).index
    
    print('=' * 60)
    print('  SECTION 3 — PREDICTION PIPELINE DEMO')
    print('  3 real test-set examples through fine-tuned RoBERTa')
    print('=' * 60)
    
    results = []
    
    for idx in io_idx:
        print(f'\nExample — IO tweet')
        r = predict_post(X_test_text[idx], y_test[idx])
        results.append(r)
    
    for idx in organic_idx:
        print(f'\nExample — organic tweet')
        r = predict_post(X_test_text[idx], y_test[idx])
        results.append(r)
    
    # ── Summary table ────────────────────────────────────────
    print('\n  Summary:')
    print('  ┌─────────┬──────────┬───────────┬────────────┬─────────┐')
    print('  │ Example │ Real     │ Predicted │ Confidence │ Correct │')
    print('  ├─────────┼──────────┼───────────┼────────────┼─────────┤')
    for i, (pred, real, conf, ok) in enumerate(results, 1):
        print(f'  │ {i}       │ {real:<8} │ {pred:<9} │ {conf:.4f}     │ {"Yes" if ok else "No":>7} │')
    print('  └─────────┴──────────┴───────────┴────────────┴─────────┘')
    
    n_correct = sum(1 for _, _, _, ok in results if ok)
    print(f'\n  {n_correct}/{len(results)} predictions correct.')
    if n_correct < len(results):
        print('  Note: Misclassifications may indicate the model struggles with')
        print('  tweets that lack strong stylistic IO signals or organic tweets')
        print('  that happen to discuss political topics similar to IO content.')


In [ ]:
# Section 4 — Adversarial Attack Setup

import os, subprocess, sys

# Clone fork if not already present
if not os.path.isdir('/kaggle/working/OpenAttack'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/LPukarowski/OpenAttack.git', '/kaggle/working/OpenAttack'],
        check=True
    )

# Clear stale module cache
for key in list(sys.modules.keys()):
    if 'OpenAttack' in key:
        del sys.modules[key]

# Load directly from source — no pip install needed
sys.path.insert(0, '/kaggle/working/OpenAttack')

import OpenAttack as oa
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import ssl, nltk

ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('wordnet', quiet=True)

print(f"OpenAttack loaded from: {oa.__file__}")
print(f"Available attackers:    {[a for a in dir(oa.attackers) if not a.startswith('_')]}")
print(f"GPUs available:         {torch.cuda.device_count()}")


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 4a budget sweep')
else:
    # Section 4a — Attack Budget Sweep (NLP equivalent of epsilon sweep)
    # BERT-Attack max_percent controls what fraction of words can be substituted.
    # 0% = no attack (baseline accuracy)
    # 10%, 20%, 40% = increasing attack budgets
    # This is the NLP equivalent of testing epsilon = 0.00, 0.01, 0.05, 0.10
    
    import tqdm, pandas as pd, numpy as np
    import matplotlib.pyplot as plt
    
    # Load 200-sample eval set (balanced IO/Organic)
    test_df_sweep = pd.read_csv('/kaggle/working/SRTY3009-IODetection/test_set.csv')
    test_sample_sweep = test_df_sweep.sample(n=200, random_state=99).reset_index(drop=True)
    
    sweep_dataset = [
        {"x": row['text'], "y": row['Label'] if 'Label' in row else row['label']}
        for _, row in test_sample_sweep.iterrows()
    ]
    
    # Reuse DisInfoClassifier from cell 5 (model_attack / tokenizer_attack must be loaded)
    # If running this cell standalone, reload:
    if 'classifier' not in dir():
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch
        model_name = 'Banjomenny/DisInfoBERT'
        tokenizer_attack = AutoTokenizer.from_pretrained(model_name)
        model_attack = AutoModelForSequenceClassification.from_pretrained(model_name).cuda()
        model_attack.eval()
    
        class DisInfoClassifier(oa.Classifier):
            def get_pred(self, input_):
                return [1 if p > 0.5 else 0 for p in self._get_io_probs(input_)]
            def get_prob(self, input_):
                probs = self._get_io_probs(input_)
                return np.array([[1 - p, p] for p in probs])
            def _get_io_probs(self, input_):
                device = next(model_attack.parameters()).device
                encodings = tokenizer_attack(list(input_), truncation=True, padding=True,
                                             max_length=128, return_tensors='pt').to(device)
                with torch.no_grad():
                    outputs = model_attack(**encodings)
                return torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy().tolist()
        classifier = DisInfoClassifier()
    
    # ── Sweep over 4 budget levels ────────────────────────────────────────────
    # Epsilon analogy: 0.00=no attack, 0.10=10% words, 0.20=20%, 0.40=40%
    budgets = [0.0, 0.1, 0.2, 0.4]
    budget_labels = ['0.00 (no attack)', '0.10', '0.20', '0.40']
    results_sweep = []
    
    # 0% budget = clean accuracy (no attack needed)
    clean_preds = classifier.get_pred([row['text'] for row in sweep_dataset])
    clean_acc = sum(p == row['y'] for p, row in zip(clean_preds, sweep_dataset)) / len(sweep_dataset)
    results_sweep.append({'budget': 0.0, 'label': '0.00 (no attack)',
                           'accuracy': clean_acc, 'attack_success_rate': 0.0})
    print(f"Budget 0.00 (clean): accuracy = {clean_acc:.3f}")
    
    for budget in budgets[1:]:
        attacker_sweep = oa.attackers.BERTAttacker(max_percent=budget)
        attack_eval_sweep = oa.AttackEval(attacker_sweep, classifier)
    
        n_success = 0
        n_correct_after = 0
        n_total = len(sweep_dataset)
    
        for result in tqdm.tqdm(attack_eval_sweep.ieval(sweep_dataset), total=n_total,
                                 desc=f'Budget {budget:.2f}'):
            orig_correct = (classifier.get_pred([result['data']['x']])[0] == result['data']['y'])
            if result['success']:
                n_success += 1
                # After successful attack, model was flipped
            else:
                if orig_correct:
                    n_correct_after += 1
    
        # Accuracy after attack = samples where model still correct
        # = originally correct samples that weren't successfully attacked
        orig_correct_count = sum(
            1 for row in sweep_dataset
            if classifier.get_pred([row['x']])[0] == row['y']
        )
        acc_after = (orig_correct_count - n_success) / n_total
        asr = n_success / n_total
    
        results_sweep.append({
            'budget': budget,
            'label': f'{budget:.2f}',
            'accuracy': max(acc_after, 0.0),
            'attack_success_rate': asr
        })
        print(f"Budget {budget:.2f}: accuracy = {acc_after:.3f}, attack success rate = {asr:.3f}")
    
    # ── Print table ───────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  SECTION 4 — ATTACK BUDGET SWEEP RESULTS")
    print(f"  (NLP equivalent of epsilon sweep)")
    print(f"{'='*60}")
    print(f"  {'Budget (max_percent)':22} {'Accuracy':10} {'Attack Success Rate':20}")
    print(f"  {'-'*55}")
    for r in results_sweep:
        print(f"  {r['label']:22} {r['accuracy']:.4f}     {r['attack_success_rate']:.4f}")
    
    # ── Accuracy vs budget chart ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    x_vals = [r['budget'] for r in results_sweep]
    y_vals = [r['accuracy'] for r in results_sweep]
    
    ax.plot(x_vals, y_vals, 'o-', color='#e05c5c', linewidth=2, markersize=8)
    ax.set_xlabel('Attack Budget (max_percent of words substituted)\n(NLP equivalent of epsilon)', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title('Section 4 — BERT-Attack: Accuracy vs Attack Budget\nDisInfoBERT (pre-defence)', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(budgets)
    ax.set_xticklabels(['0.00\n(no attack)', '0.10', '0.20', '0.40'])
    ax.grid(True, alpha=0.3)
    
    for x, y in zip(x_vals, y_vals):
        ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center')
    
    plt.tight_layout()
    plt.savefig('section4_accuracy_vs_budget.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: section4_accuracy_vs_budget.png')
    
    # Store for use in defence comparison cell
    sweep_pre_defence = results_sweep


In [ ]:
import os

AUGMENTED_CSV = '/kaggle/working/SRTY3009-IODetection/augmented_training_set.csv'
FALLBACK_CSV  = '/kaggle/working/augmented_training_set.csv'

if os.path.isfile(AUGMENTED_CSV) or os.path.isfile(FALLBACK_CSV):
    found = AUGMENTED_CSV if os.path.isfile(AUGMENTED_CSV) else FALLBACK_CSV
    print(f"Augmented training set already exists at:\n  {found}")
    print("Skipping attack generation. Delete the file to re-run.")
else:
    # Cell: Adversarial Training Data Generation
    # Wraps DisInfoBERT as an OpenAttack Classifier, attacks 4050 balanced
    # training samples with BERT-Attack, and saves adversarial examples.
    
    import pandas as pd
    
    # ── Load balanced training data ──────────────────────────────────────────
    train_bal_df = pd.read_csv('/kaggle/working/SRTY3009-IODetection/train_balanced.csv')
    X_train_text_bal = train_bal_df['text']
    try:
        y_train_bal = train_bal_df['Label']
    except KeyError:
        y_train_bal = train_bal_df['label']
        
    print(f"Loaded train_balanced.csv: {len(train_bal_df):,} rows")
    print(f"  IO: {(y_train_bal == 1).sum():,}  |  Organic: {(y_train_bal == 0).sum():,}")
    
    # ── LOAD MODEL ───────────────────────────────────────────────────────────
    model_name = 'Banjomenny/DisInfoBERT'
    tokenizer_attack = AutoTokenizer.from_pretrained(model_name)
    model_attack = AutoModelForSequenceClassification.from_pretrained(model_name)
    model_attack = model_attack.cuda()
    model_attack.eval()
    
    # ── OPENATTACK CLASSIFIER WRAPPER ────────────────────────────────────────
    # oa.Classifier is the correct base class in OpenAttack 2.x
    class DisInfoClassifier(oa.Classifier):
        def get_pred(self, input_):
            return [1 if p > 0.5 else 0 for p in self._get_io_probs(input_)]
    
        def get_prob(self, input_):
            probs = self._get_io_probs(input_)
            return np.array([[1 - p, p] for p in probs])
    
        def _get_io_probs(self, input_):
            device = next(model_attack.parameters()).device
            encodings = tokenizer_attack(
                list(input_),
                truncation=True,
                padding=True,
                max_length=128,
                return_tensors='pt'
            ).to(device)
            with torch.no_grad():
                outputs = model_attack(**encodings)
                probs = torch.softmax(outputs.logits, dim=-1)
                return probs[:, 1].cpu().numpy().tolist()
    
    classifier = DisInfoClassifier()
    
    # ── PREPARE TRAINING SAMPLES ─────────────────────────────────────────────
    import pandas as pd
    
    train_df = pd.DataFrame({
        'text':  X_train_text_bal.tolist(),
        'label': y_train_bal.tolist()
    })
    
    io_samples  = train_df[train_df['label'] == 1].sample(2025, random_state=42)
    org_samples = train_df[train_df['label'] == 0].sample(2025, random_state=42)
    attack_samples = pd.concat([io_samples, org_samples]).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)
    
    print(f"Attacking {len(attack_samples)} training samples")
    print(f"  IO:      {(attack_samples['label'] == 1).sum()}")
    print(f"  Organic: {(attack_samples['label'] == 0).sum()}")
    
    # OpenAttack 2.x dataset format: list of {"x": text, "y": label}
    train_attack_dataset = [
        {"x": row['text'], "y": row['label']}
        for _, row in attack_samples.iterrows()
    ]
    
    # ── RUN BERT-ATTACK ───────────────────────────────────────────────────────
    # Use ieval() not eval() — eval() only prints a summary and returns nothing.
    # ieval() yields per-sample result dicts with keys: success, result, data
    attacker = oa.attackers.BERTAttacker()
    attack_eval = oa.AttackEval(attacker, classifier)
    
    print("\nRunning BERT-Attack on 4050 training samples (this will take a while)...")
    
    # ── EXTRACT ADVERSARIAL EXAMPLES ─────────────────────────────────────────
    adversarial_examples = []
    n_total = len(train_attack_dataset)
    
    import tqdm
    for result in tqdm.tqdm(attack_eval.ieval(train_attack_dataset), total=n_total):
        if result["success"]:
            adversarial_examples.append({
                'text':  result["result"],       # adversarial text
                'Label': result["data"]["y"]     # original label preserved
            })
    
    df_adversarial = pd.DataFrame(adversarial_examples)
    n_success = len(df_adversarial)
    n_total   = len(attack_samples)
    
    print(f"\n{'='*50}")
    print(f"  ATTACK RESULTS")
    print(f"{'='*50}")
    print(f"  Total samples attacked : {n_total}")
    print(f"  Successful attacks     : {n_success}")
    print(f"  Failed attacks         : {n_total - n_success}")
    print(f"  Success rate           : {n_success / n_total * 100:.1f}%")
    
    # ── AUGMENT TRAINING SET ──────────────────────────────────────────────────
    df_train_original = pd.DataFrame({
        'text':  X_train_text_bal.tolist(),
        'Label': y_train_bal.tolist()
    })
    
    df_augmented = pd.concat([
        df_train_original,
        df_adversarial
    ]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\n{'='*50}")
    print(f"  AUGMENTED TRAINING SET")
    print(f"{'='*50}")
    print(f"  Original:    {len(df_train_original):,}")
    print(f"  Adversarial: {n_success:,}")
    print(f"  Total:       {len(df_augmented):,}")
    print(f"  Adv %:       {n_success / len(df_augmented) * 100:.1f}%")
    
    df_adversarial.to_csv('/kaggle/working/adversarial_examples.csv', index=False)
    df_augmented.to_csv('/kaggle/working/augmented_training_set.csv', index=False)
    print(f"\n  Saved to /kaggle/working/ ✅  Ready for adversarial training")

In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 4 retrain')
else:
    %%time
    # Cell: Adversarial Training — Retrain DisInfoBERT on Augmented Dataset
    
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        TrainingArguments,
        Trainer
    )
    from sklearn.metrics import f1_score, accuracy_score
    from datasets import Dataset as HFDataset
    import numpy as np
    import pandas as pd
    import os
    
    print("Retraining on augmented dataset with adversarial examples...")
    
    # ── Load augmented set (from CSV if session was interrupted) ─────────────
    AUGMENTED_CSV = '/kaggle/working/SRTY3009-IODetection/augmented_training_set.csv'
    FALLBACK_CSV  = '/kaggle/working/augmented_training_set.csv'

    if 'df_augmented' not in dir():
        if os.path.isfile(AUGMENTED_CSV):
            df_augmented = pd.read_csv(AUGMENTED_CSV)
            print(f"Loaded augmented set from repo dir: {len(df_augmented):,} rows")
        elif os.path.isfile(FALLBACK_CSV):
            df_augmented = pd.read_csv(FALLBACK_CSV)
            print(f"Loaded augmented set from working dir: {len(df_augmented):,} rows")
        else:
            raise FileNotFoundError(
                "augmented_training_set.csv not found in either:\n"
                f"  {AUGMENTED_CSV}\n  {FALLBACK_CSV}\n"
                "Run attack_only mode first to generate it."
            )
    
    # ── Reload tokenizer (may differ from attack tokenizer in fresh sessions) ─
    model_name = 'Banjomenny/DisInfoBERT'
    tokenizer_train = AutoTokenizer.from_pretrained(model_name)
    
    # ── Tokenize ──────────────────────────────────────────────────────────────
    print("Tokenizing augmented training set...")
    
    def tokenize_batch(texts, labels, tokenizer, max_length=128):
        encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length
        )
        data = dict(encodings)          # BatchEncoding → plain dict for from_dict()
        data['labels'] = labels
        return HFDataset.from_dict(data)
    
    train_aug_dataset = tokenize_batch(
        df_augmented['text'].tolist(),
        (df_augmented['Label'] if 'Label' in df_augmented.columns else df_augmented['label']).tolist(),
        tokenizer_train
    )
    
    # ── Load test set ─────────────────────────────────────────────────────────
    test_df = pd.read_csv('/kaggle/working/SRTY3009-IODetection/test_set.csv')
    X_test_text = test_df['text']
    try:
        y_test = test_df['Label']
    except KeyError:
        y_test = test_df['label']
        print(f"Loaded test_set.csv: {len(test_df):,} rows")
    
    test_aug_dataset = tokenize_batch(
        X_test_text.tolist(),
        y_test.tolist(),
        tokenizer_train
    )
    
    # ── Fresh model instance ──────────────────────────────────────────────────
    model_defended = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {
            'f1':       f1_score(labels, predictions, average='weighted'),
            'accuracy': accuracy_score(labels, predictions)
        }
    
    training_args = TrainingArguments(
        output_dir='./results_defended',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',       # 'evaluation_strategy' deprecated in HF >= 4.x
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        fp16=True,
        logging_steps=100,
        report_to='none'
    )
    
    trainer_defence = Trainer(
        model=model_defended,
        args=training_args,
        train_dataset=train_aug_dataset,
        eval_dataset=test_aug_dataset,
        compute_metrics=compute_metrics
    )
    
    trainer_defence.train()
    
    # Save defended model
    save_path = '/kaggle/working/DisInfoBERT-defended'
    trainer_defence.save_model(save_path)
    tokenizer_train.save_pretrained(save_path)
    print(f"Defended model saved to {save_path} ✓")


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 4 evaluate defended')
else:
    # Cell: Evaluate Defended Model — BERT-Attack on 100 test samples
    # Loads the adversarially-trained DisInfoBERT-defended and re-runs the attack
    # to measure robustness improvement vs the original model.
    
    import os, tqdm
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import OpenAttack as oa
    import torch
    import numpy as np
    import pandas as pd
    
    # ── Load defended model ───────────────────────────────────────────────────
    defended_path = '/kaggle/working/DisInfoBERT-defended'
    tokenizer_def = AutoTokenizer.from_pretrained(defended_path)
    model_def = AutoModelForSequenceClassification.from_pretrained(defended_path)
    model_def = model_def.cuda()
    model_def.eval()
    print(f"Loaded defended model from {defended_path}")
    
    # ── Classifier wrapper ────────────────────────────────────────────────────
    class DefendedClassifier(oa.Classifier):
        def get_pred(self, input_):
            return [1 if p > 0.5 else 0 for p in self._get_io_probs(input_)]
    
        def get_prob(self, input_):
            probs = self._get_io_probs(input_)
            return np.array([[1 - p, p] for p in probs])
    
        def _get_io_probs(self, input_):
            device = next(model_def.parameters()).device
            encodings = tokenizer_def(
                list(input_),
                return_tensors='pt',
                truncation=True,
                padding=True,
                max_length=128
            ).to(device)
            with torch.no_grad():
                outputs = model_def(**encodings)
            probs = torch.softmax(outputs.logits, dim=-1)
            return probs[:, 1].cpu().numpy().tolist()
    
    victim = DefendedClassifier()
    
    # ── Sample 100 from test set ──────────────────────────────────────────────
    test_df = pd.read_csv('/kaggle/working/SRTY3009-IODetection/test_set.csv')
    test_sample = test_df.sample(n=100, random_state=42).reset_index(drop=True)
    
    test_attack_dataset = [
        {"x": row['text'], "y": row['Label'] if 'Label' in row else row['label']}
        for _, row in test_sample.iterrows()
    ]
    
    # ── Run BERT-Attack with progress bar ────────────────────────────────────
    attacker = oa.attackers.BERTAttacker()
    attack_eval = oa.AttackEval(attacker, victim)
    
    print("\nRunning BERT-Attack on defended model (100 samples)...")
    
    successes, failures = [], []
    for result in tqdm.tqdm(attack_eval.ieval(test_attack_dataset), total=len(test_attack_dataset)):
        if result["success"]:
            successes.append(result["result"])
        else:
            failures.append(result["data"]["x"])
    
    n_total   = len(test_attack_dataset)
    n_success = len(successes)
    
    print(f"\n{'='*50}")
    print(f"  DEFENDED MODEL ATTACK RESULTS")
    print(f"{'='*50}")
    print(f"  Total samples      : {n_total}")
    print(f"  Successful attacks : {n_success}")
    print(f"  Failed attacks     : {n_total - n_success}")
    print(f"  Attack success rate: {n_success / n_total * 100:.1f}%")
    print(f"  Robustness rate    : {(n_total - n_success) / n_total * 100:.1f}%")
    print("\nAttack evaluation complete.")


In [ ]:
if RUN_MODE != 'full':
    print('Skipping: Section 4 defence comparison')
else:
    # Section 4 — Before vs After Defence Comparison Table
    # Compares original DisInfoBERT vs adversarially-trained DisInfoBERT-defended
    # at attack budget = 0.20 (equivalent to epsilon=0.05)
    
    import pandas as pd
    import matplotlib.pyplot as plt
    import numpy as np
    
    # ── Run attack on ORIGINAL model at budget=0.20 ──────────────────────────
    # (reuse sweep results if available, otherwise re-run)
    if 'sweep_pre_defence' in dir():
        pre_acc  = next(r['accuracy'] for r in sweep_pre_defence if r['budget'] == 0.2)
        pre_asr  = next(r['attack_success_rate'] for r in sweep_pre_defence if r['budget'] == 0.2)
    else:
        print("sweep_pre_defence not found — re-run the attack budget sweep cell first.")
        pre_acc, pre_asr = None, None
    
    # Post-defence: use results from cell 7 (n_success, n_total already computed)
    if 'n_success' in dir() and 'n_total' in dir():
        post_asr = n_success / n_total
        post_acc = (n_total - n_success) / n_total   # approximate
    else:
        print("Cell 7 results not found — re-run defended model evaluation cell first.")
        post_asr, post_acc = None, None
    
    # ── Comparison table ──────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  SECTION 4 — DEFENCE COMPARISON (attack budget = 0.20)")
    print(f"  Defence method: Adversarial Training on augmented dataset")
    print(f"{'='*65}")
    print(f"  {'Metric':30} {'Before Defence':16} {'After Defence':16} {'Change':10}")
    print(f"  {'-'*65}")
    
    if pre_acc is not None and post_acc is not None:
        rows = [
            ('Accuracy under attack',     pre_acc,  post_acc,  post_acc - pre_acc),
            ('Attack success rate',        pre_asr,  post_asr,  post_asr - pre_asr),
        ]
        for label, before, after, delta in rows:
            direction = 'improved' if (label == 'Accuracy under attack' and delta > 0) or \
                                       (label == 'Attack success rate'   and delta < 0) else 'worsened'
            print(f"  {label:30} {before:.4f}           {after:.4f}           {delta:+.4f} ({direction})")
    else:
        print("  (Run sweep and defence cells to populate this table)")
    
    # ── Bar chart: before vs after ────────────────────────────────────────────
    if pre_acc is not None and post_acc is not None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle('Section 4 — Defence Impact: Before vs After Adversarial Training', fontsize=13)
    
        categories = ['Before Defence', 'After Defence']
        colors = ['#e05c5c', '#5c9ee0']
    
        axes[0].bar(categories, [pre_acc, post_acc], color=colors)
        axes[0].set_title('Accuracy Under Attack (budget=0.20)')
        axes[0].set_ylim(0, 1.0)
        axes[0].set_ylabel('Accuracy')
        for i, v in enumerate([pre_acc, post_acc]):
            axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
    
        axes[1].bar(categories, [pre_asr, post_asr], color=colors)
        axes[1].set_title('Attack Success Rate (budget=0.20)')
        axes[1].set_ylim(0, 1.0)
        axes[1].set_ylabel('Attack Success Rate')
        for i, v in enumerate([pre_asr, post_asr]):
            axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
    
        plt.tight_layout()
        plt.savefig('section4_defence_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: section4_defence_comparison.png')


In [ ]:
# Cleanup — remove cloned repo from sys.path and working dir if no longer needed
import sys

sys.path = [p for p in sys.path if 'OpenAttack' not in p]

for key in list(sys.modules.keys()):
    if 'OpenAttack' in key:
        del sys.modules[key]

print("OpenAttack removed from import path and module cache.")
# Uncomment to also delete from disk (frees ~200MB):
# !rm -rf /kaggle/working/OpenAttack
